# Experiment 7: Text Summarization using Large Language Models (LLM)
## Notebook 2: Prompt Engineering, Model Inference & ROUGE Evaluation

This notebook demonstrates:
1. Abstractive text summarization using pre-trained Transformer LLMs (BART / DistilBART).
2. Prompt engineering variations (`v1`, `v2`, `best_prompt`).
3. Extractive baseline comparison using TF-IDF / TextRank.
4. Quantitative evaluation using ROUGE (ROUGE-1, ROUGE-2, ROUGE-L) and BERTScore.

In [ ]:
import os
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
sys.path.insert(0, os.path.join(PROJECT_ROOT, "src"))

from data_loader import get_or_create_raw_articles, split_dataset
from data_preprocessing import preprocess_corpus
from prompt_templates import PromptManager, create_summarization_prompt
from summarizer import AbstractiveSummarizer, ExtractiveSummarizer, summarize_with_prompt_engineering
from evaluation import calculate_rouge_scores, calculate_bertscore, evaluate_summary_quality
from visualization import plot_prompt_comparison, plot_abstractive_vs_extractive

### 1. Load Pre-trained Summarization Models

In [ ]:
# Initialize Abstractive LLM and Extractive Baseline
abstractive_model = AbstractiveSummarizer(model_name="sshleifer/distilbart-cnn-12-6", device="auto")
extractive_model = ExtractiveSummarizer(num_sentences=2)
prompt_mgr = PromptManager(prompt_dir=os.path.join(PROJECT_ROOT, "models", "prompts"))
print("Available Prompt Templates:", prompt_mgr.get_prompt_keys())

### 2. Run Inference on a Sample Technical Article

In [ ]:
raw_csv = os.path.join(PROJECT_ROOT, "data", "raw", "articles.csv")
df = preprocess_corpus(get_or_create_raw_articles(raw_csv))
sample_doc = df.iloc[0]

print("=" * 60)
print(f"ARTICLE TITLE: {sample_doc['title']}")
print("=" * 60)
print("ORIGINAL TEXT:")
print(sample_doc["document_text"])
print("\nGROUND TRUTH REFERENCE:")
print(sample_doc["reference_summary"])

### 3. Compare Abstractive LLM Generation vs Extractive Baseline

In [ ]:
# Abstractive summary with best prompt
abs_out = summarize_with_prompt_engineering(sample_doc["document_text"], prompt_template_key="best", summarizer=abstractive_model)
ext_out = extractive_model.summarize(sample_doc["document_text"])

print("\n--- GENERATED ABSTRACTIVE SUMMARY ---")
print(abs_out["summary_text"])
print("\n--- EXTRACTIVE BASELINE SUMMARY ---")
print(ext_out)

# Evaluate both
abs_eval = evaluate_summary_quality(sample_doc["reference_summary"], abs_out["summary_text"], sample_doc["document_text"])
ext_eval = evaluate_summary_quality(sample_doc["reference_summary"], ext_out, sample_doc["document_text"])

pd.DataFrame([
    {"Method": "Abstractive (LLM)", "ROUGE-1": abs_eval["rouge1_f1"], "ROUGE-2": abs_eval["rouge2_f1"], "ROUGE-L": abs_eval["rougeL_f1"], "BERTScore": abs_eval["bertscore_f1"], "Words": abs_eval["summary_words"]},
    {"Method": "Extractive (TF-IDF)", "ROUGE-1": ext_eval["rouge1_f1"], "ROUGE-2": ext_eval["rouge2_f1"], "ROUGE-L": ext_eval["rougeL_f1"], "BERTScore": ext_eval["bertscore_f1"], "Words": ext_eval["summary_words"]}
])

### 4. Prompt Engineering Variation Comparison

In [ ]:
prompt_results = []
for p_key in ["v1", "v2", "best"]:
    res = summarize_with_prompt_engineering(sample_doc["document_text"], prompt_template_key=p_key, summarizer=abstractive_model)
    q = evaluate_summary_quality(sample_doc["reference_summary"], res["summary_text"])
    prompt_results.append({
        "prompt_version": f"Prompt {p_key}",
        "rouge1_f1": q["rouge1_f1"],
        "rouge2_f1": q["rouge2_f1"],
        "rougeL_f1": q["rougeL_f1"],
        "bertscore_f1": q["bertscore_f1"],
        "latency_seconds": res["latency_seconds"],
        "summary_text": res["summary_text"]
    })

df_prompts = pd.DataFrame(prompt_results)
df_prompts[["prompt_version", "rouge1_f1", "rouge2_f1", "rougeL_f1", "bertscore_f1", "latency_seconds"]]